# Phase 1: Data Acquisition and Cleaning
This notebook handles loading the raw Yellow Taxi parquet files, addressing missing values, and filtering anomalies based on the data dictionary.

In [1]:
import pandas as pd
import glob
import os
from pathlib import Path

# Configure pandas to show more columns
pd.set_option("display.max_columns", 50)

## 1. Load `.parquet` files from `data/raw/`

In [2]:
raw_data_dir = Path("../data/raw")
parquet_files = sorted(raw_data_dir.glob("yellow_tripdata_*.parquet"))

print(f"Found {len(parquet_files)} parquet files.")

def load_data(files):
    dfs = [pd.read_parquet(f) for f in files]
    return pd.concat(dfs, ignore_index=True)

# Loading the first month to work with a representative sample, or use all if memory allows.
df = load_data(parquet_files)
print(f"Data shape: {df.shape}")
df.head()

Found 14 parquet files.
Data shape: (55847357, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


## 2. Handle missing values and anomalies in time variables

In [3]:
# Check for missing time variables
print("Missing pickup times:", df['tpep_pickup_datetime'].isna().sum())
print("Missing dropoff times:", df['tpep_dropoff_datetime'].isna().sum())

# Drop rows with missing timestamps (if any)
df = df.dropna(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])

# Ensure time variables are datetime objects (already handled by parquet but good to ensure)
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# Calculate trip duration in minutes
df['trip_duration'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60.0

# Filter anomalous times:
# 1. Negative or zero trip durations
# 2. Extremely long trips (e.g., > 24 hours)
# 3. Trips with pickup years not matching our dataset bounds (2024 - 2026)

initial_len = len(df)
# Allowed years 2024, 2025, 2026
valid_years = [2025, 2026]
df = df[
    (df['trip_duration'] > 0) &
    (df['trip_duration'] < 24 * 60) &
    (df['tpep_pickup_datetime'].dt.year.isin(valid_years))
]

print(f"Removed {initial_len - len(df)} records with anomalous time variables.")

Missing pickup times: 0
Missing dropoff times: 0
Removed 632425 records with anomalous time variables.


## 3. Filter invalid coordinates, extreme passenger counts, or negative fares
Based on the TLC data dictionary:
- `passenger_count` should be > 0 and reasonable (let's say <= 9).
- `fare_amount`, `total_amount`, `trip_distance` must be strictly positive.
- `PULocationID` and `DOLocationID` should map to known TLC zones (1 to 263).

In [4]:
initial_len = len(df)

# 1. Filter negative or zero fares and distances
df = df[
    (df['fare_amount'] > 0) &
    (df['total_amount'] > 0) &
    (df['trip_distance'] > 0)
]

# 2. Filter passenger counts
# Since passenger_count might be NaN for some vendor types or system fails,
# we can either drop NaNs or impute. For strict cleaning, let's filter those with 0 or > 9
# and allow NaNs or fill them with median (1).
df['passenger_count'] = df['passenger_count'].fillna(1)
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] <= 9)]

# 3. Validate Locations (TLC zones range from 1 to 263, 264/265 are unknown/NA)
valid_zones = set(range(1, 264))
df = df[
    (df['PULocationID'].isin(valid_zones)) &
    (df['DOLocationID'].isin(valid_zones))
]

print(f"Removed {initial_len - len(df)} records due to invalid fares, passenger counts, or locations.")
print(f"Final Data Shape: {df.shape}")

Removed 4956834 records due to invalid fares, passenger counts, or locations.
Final Data Shape: (50258098, 21)


## 4. Save Cleaned Data
Save the cleaned dataframe to an intermediate or processed directory for the next phases.

In [5]:
# We will create a interim data directory and save to parquet.
interim_dir = Path("../data/interim")
interim_dir.mkdir(parents=True, exist_ok=True)

# Save a subset or the whole processed df
interim_file = interim_dir / "cleaned_data.parquet"
df.to_parquet(interim_file, index=False)
print(f"Cleaned data successfully saved to {interim_file}")

Cleaned data successfully saved to ..\data\interim\cleaned_data.parquet
